# Customer Personality Analysis - Customer Segmentation

This notebook performs Exploratory Data Analysis and Customer Segmentation using Principal Component Analysis (PCA) and K-Means clustering. We use the custom pipeline defined in `src/` modules.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_processing import preprocess_pipeline
from src.clustering import run_clustering_pipeline, find_optimal_k

# Set style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 1. Load and Preprocess the Dataset

We load and preprocess the dataset using the module `src/data_processing.py`.

> **Note**: Please ensure that `data/marketing_campaign.csv` exists. If you haven't downloaded it yet, run `./download_dataset.sh` or download it manually from Kaggle and place it in the `data/` folder.

In [ ]:
data_path = os.path.join("..", "data", "marketing_campaign.csv")
if not os.path.exists(data_path):
    print(f"Error: {data_path} not found. Please download the dataset first.")
else:
    df_processed = preprocess_pipeline(data_path)
    print("\nPreprocessed data shape:", df_processed.shape)
    display(df_processed.head())

## 2. Determine Optimal Clusters (Elbow Method)

We use the elbow method to decide the number of clusters $K$ by evaluating K-Means inertia on the PCA-reduced data.

In [ ]:
if os.path.exists(data_path):
    # Scale and prepare data
    from src.clustering import prepare_data_for_clustering, apply_pca
    scaled_data, _, _ = prepare_data_for_clustering(df_processed)
    pca_data, _ = apply_pca(scaled_data, n_components=3)
    
    # Calculate inertias
    k_range, inertias = find_optimal_k(pca_data, max_k=10)
    
    # Plot Elbow curve
    plt.figure(figsize=(8, 5))
    plt.plot(k_range, inertias, 'bx-')
    plt.xlabel('Number of Clusters (k)')
    plt.ylabel('Inertia')
    plt.title('The Elbow Method showing the optimal k')
    plt.show()

## 3. Run Clustering Pipeline

Based on the elbow plot, 4 clusters are typically chosen. Let's run the pipeline and add the cluster labels back to our dataset.

In [ ]:
if os.path.exists(data_path):
    df_clustered, pca_data, preprocessor, pca, kmeans = run_clustering_pipeline(
        df_processed, n_components=3, n_clusters=4
    )
    print("Clustered data columns:", df_clustered.columns.tolist())
    print("\nCluster counts:")
    print(df_clustered['Cluster'].value_counts().sort_index())

## 4. Visualize the Clusters in PCA Space

Let's visualize the clusters using the first two principal components.

In [ ]:
if os.path.exists(data_path):
    plt.figure(figsize=(10, 8))
    sns.scatterplot(
        x='PCA_Component_1', 
        y='PCA_Component_2', 
        hue='Cluster', 
        palette='viridis', 
        data=df_clustered, 
        alpha=0.8
    )
    plt.title('Customer Segments Visualized with PCA Components')
    plt.xlabel('Principal Component 1')
    plt.ylabel('Principal Component 2')
    plt.legend(title='Cluster')
    plt.show()

## 5. Profiling Customer Segments

Let's analyze the averages and distributions of key features across clusters to understand the personality of each segment.

In [ ]:
if os.path.exists(data_path):
    # Calculate mean stats for each cluster
    cluster_profile = df_clustered.groupby('Cluster')[[
        'Age', 'Income', 'Total_Spend', 'Children', 'Total_Purchases'
    ]].mean()
    print("Cluster Profiling (Averages):")
    display(cluster_profile)
    
    # Plot boxplots of key features
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    
    sns.boxplot(ax=axes[0], x='Cluster', y='Income', data=df_clustered, palette='viridis')
    axes[0].set_title('Income Distribution per Cluster')
    
    sns.boxplot(ax=axes[1], x='Cluster', y='Total_Spend', data=df_clustered, palette='viridis')
    axes[1].set_title('Total Spend Distribution per Cluster')
    
    sns.boxplot(ax=axes[2], x='Cluster', y='Age', data=df_clustered, palette='viridis')
    axes[2].set_title('Age Distribution per Cluster')
    
    plt.tight_layout()
    plt.show()